# 04 Win Rate — Logistic Regression

## Goal

Picks up from the screening pass in `03_exploratory_analysis.ipynb`: for a hand-picked list of "cards of interest" (fill in `CARDS_OF_INTEREST` below based on what that screen surfaced), restrict to occasions where the card was offered and fit

```
victory ~ was_picked + hp_ratio + floor + relic_count + ascension_level
```

`was_picked`'s coefficient is the card's effect on win probability *holding run state at the time of the offer constant* — a cleaner signal than the raw pick/no-pick lift computed in 03, which doesn't control for anything.

**Deliberately excluded:** `floor_reached` and `floors_gained` are not covariates here, even though they're in the table. Both are facts about how the run *ended*, not facts known at the time of the pick — `floor_reached` is close to deterministic of `victory` (a run that reaches floor 57 essentially won), so including it would leak the outcome into the predictors rather than control for a legitimate confounder. `floor` (the pick's own floor, i.e. `choice_floor`) is fine to include — that's "how far into the run this decision happened," known at decision time.

In [ ]:
import os
from pathlib import Path

# Jupyter's cwd is the notebook's directory; project root is one level up.
PROJECT_ROOT = Path.cwd().parent

os.environ["JAVA_HOME"] = str(PROJECT_ROOT / ".jdk17" / "jdk-17.0.20+8")
os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["PATH"] = str(PROJECT_ROOT / ".venv" / "Scripts") + os.pathsep + r"C:\hadoop\bin" + os.pathsep + os.environ["PATH"]
os.environ["PYSPARK_PYTHON"] = str(PROJECT_ROOT / ".venv" / "Scripts" / "python.exe")
os.environ["PYSPARK_DRIVER_PYTHON"] = str(PROJECT_ROOT / ".venv" / "Scripts" / "python.exe")

GOLD_CARD_CHOICE_EVENTS_PATH = str(PROJECT_ROOT / "raw_data" / "gold" / "card_choice_events")

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from delta import configure_spark_with_delta_pip
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

builder = (
    SparkSession.builder.master("local[*]")
    .appName("win-rate-logistic-regression")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.driver.memory", "16g")
    .config("spark.sql.shuffle.partitions", "100")
)
spark = configure_spark_with_delta_pip(builder).getOrCreate()

df = spark.read.format("delta").load(GOLD_CARD_CHOICE_EVENTS_PATH)
print(f"Loaded {GOLD_CARD_CHOICE_EVENTS_PATH}")

## Cards of interest

Fill this in from `03_exploratory_analysis.ipynb`'s screening output — the cards that showed up with high `floors_gained_lift` and/or `win_rate_lift` (and enough sample size to trust the estimate). One row per (character, card) pair.

In [ ]:
CARDS_OF_INTEREST = pd.DataFrame([
    # Top 40 per character by win_rate_lift from 03_exploratory_analysis.ipynb's screening
    # (MIN_GROUP_SIZE=500 in both picked/not-picked groups), regenerated 2026-08-17.
    ("DEFECT", "Master of Strategy"),
    ("DEFECT", "Master of Strategy+1"),
    ("DEFECT", "Thinking Ahead"),
    ("DEFECT", "Secret Technique"),
    ("DEFECT", "Secret Weapon"),
    ("DEFECT", "Seek+1"),
    ("DEFECT", "Violence"),
    ("DEFECT", "Purity"),
    ("DEFECT", "Reboot+1"),
    ("DEFECT", "Defragment+1"),
    ("DEFECT", "Heatsinks+1"),
    ("DEFECT", "Biased Cognition+1"),
    ("DEFECT", "Skim+1"),
    ("DEFECT", "Impatience+1"),
    ("DEFECT", "Impatience"),
    ("DEFECT", "Force Field+1"),
    ("DEFECT", "Secret Technique+1"),
    ("DEFECT", "Panache"),
    ("DEFECT", "Fission+1"),
    ("DEFECT", "Buffer+1"),
    ("DEFECT", "Seek"),
    ("DEFECT", "Biased Cognition"),
    ("DEFECT", "Coolheaded+1"),
    ("DEFECT", "Dark Shackles"),
    ("DEFECT", "Flash of Steel+1"),
    ("DEFECT", "Force Field"),
    ("DEFECT", "Deep Breath"),
    ("DEFECT", "Steam Power+1"),
    ("DEFECT", "Apotheosis"),
    ("DEFECT", "Consume+1"),
    ("DEFECT", "Skim"),
    ("DEFECT", "Glacier"),
    ("DEFECT", "Defragment"),
    ("DEFECT", "Storm+1"),
    ("DEFECT", "Recycle+1"),
    ("DEFECT", "Blizzard+1"),
    ("DEFECT", "Reboot"),
    ("DEFECT", "Finesse"),
    ("DEFECT", "Heatsinks"),
    ("DEFECT", "Chill"),
    ("IRONCLAD", "Master of Strategy"),
    ("IRONCLAD", "Secret Technique"),
    ("IRONCLAD", "Master of Strategy+1"),
    ("IRONCLAD", "Offering+1"),
    ("IRONCLAD", "Thinking Ahead"),
    ("IRONCLAD", "Secret Weapon"),
    ("IRONCLAD", "Violence"),
    ("IRONCLAD", "Secret Technique+1"),
    ("IRONCLAD", "Purity"),
    ("IRONCLAD", "Offering"),
    ("IRONCLAD", "Deep Breath+1"),
    ("IRONCLAD", "Deep Breath"),
    ("IRONCLAD", "Finesse"),
    ("IRONCLAD", "Acrobatics"),
    ("IRONCLAD", "Violence+1"),
    ("IRONCLAD", "Burning Pact+1"),
    ("IRONCLAD", "Dark Embrace+1"),
    ("IRONCLAD", "Impervious+1"),
    ("IRONCLAD", "Dark Shackles"),
    ("IRONCLAD", "Feel No Pain+1"),
    ("IRONCLAD", "Exhume+1"),
    ("IRONCLAD", "Impervious"),
    ("IRONCLAD", "Purity+1"),
    ("IRONCLAD", "Battle Trance+1"),
    ("IRONCLAD", "Shockwave+1"),
    ("IRONCLAD", "Corruption"),
    ("IRONCLAD", "Exhume"),
    ("IRONCLAD", "Shockwave"),
    ("IRONCLAD", "Corruption+1"),
    ("IRONCLAD", "Battle Trance"),
    ("IRONCLAD", "Limit Break"),
    ("IRONCLAD", "Panacea"),
    ("IRONCLAD", "Finesse+1"),
    ("IRONCLAD", "Entrench+1"),
    ("IRONCLAD", "Backflip+1"),
    ("IRONCLAD", "Apotheosis"),
    ("IRONCLAD", "Panache"),
    ("IRONCLAD", "Dark Embrace"),
    ("IRONCLAD", "Disarm+1"),
    ("IRONCLAD", "Thinking Ahead+1"),
    ("THE_SILENT", "Master of Strategy"),
    ("THE_SILENT", "Secret Technique"),
    ("THE_SILENT", "Thinking Ahead"),
    ("THE_SILENT", "Purity"),
    ("THE_SILENT", "Master of Strategy+1"),
    ("THE_SILENT", "Adrenaline+1"),
    ("THE_SILENT", "After Image+1"),
    ("THE_SILENT", "Secret Weapon"),
    ("THE_SILENT", "Violence"),
    ("THE_SILENT", "Wraith Form v2+1"),
    ("THE_SILENT", "Finesse+1"),
    ("THE_SILENT", "Adrenaline"),
    ("THE_SILENT", "Malaise+1"),
    ("THE_SILENT", "Apotheosis"),
    ("THE_SILENT", "Acrobatics+1"),
    ("THE_SILENT", "Finesse"),
    ("THE_SILENT", "After Image"),
    ("THE_SILENT", "Escape Plan+1"),
    ("THE_SILENT", "Wraith Form v2"),
    ("THE_SILENT", "Calculated Gamble+1"),
    ("THE_SILENT", "Impatience"),
    ("THE_SILENT", "Deep Breath"),
    ("THE_SILENT", "Footwork+1"),
    ("THE_SILENT", "Deep Breath+1"),
    ("THE_SILENT", "Dark Shackles"),
    ("THE_SILENT", "Reflex+1"),
    ("THE_SILENT", "Impatience+1"),
    ("THE_SILENT", "Catalyst+1"),
    ("THE_SILENT", "Burst+1"),
    ("THE_SILENT", "Malaise"),
    ("THE_SILENT", "Panache"),
    ("THE_SILENT", "Panacea+1"),
    ("THE_SILENT", "Tactician+1"),
    ("THE_SILENT", "Escape Plan"),
    ("THE_SILENT", "Doppelganger+1"),
    ("THE_SILENT", "Catalyst"),
    ("THE_SILENT", "Calculated Gamble"),
    ("THE_SILENT", "Footwork"),
    ("THE_SILENT", "Acrobatics"),
    ("THE_SILENT", "Secret Technique+1"),
    ("WATCHER", "Master of Strategy"),
    ("WATCHER", "Scrawl+1"),
    ("WATCHER", "Vault+1"),
    ("WATCHER", "Vault"),
    ("WATCHER", "Purity"),
    ("WATCHER", "Scrawl"),
    ("WATCHER", "Secret Technique"),
    ("WATCHER", "Thinking Ahead"),
    ("WATCHER", "Violence"),
    ("WATCHER", "Finesse+1"),
    ("WATCHER", "Secret Weapon"),
    ("WATCHER", "Omniscience+1"),
    ("WATCHER", "Omniscience"),
    ("WATCHER", "Fasting2+1"),
    ("WATCHER", "Fasting2"),
    ("WATCHER", "Finesse"),
    ("WATCHER", "TalkToTheHand+1"),
    ("WATCHER", "Flash of Steel"),
    ("WATCHER", "Flash of Steel+1"),
    ("WATCHER", "TalkToTheHand"),
    ("WATCHER", "Adaptation"),
    ("WATCHER", "Wish+1"),
    ("WATCHER", "EmptyMind+1"),
    ("WATCHER", "WheelKick"),
    ("WATCHER", "InnerPeace+1"),
    ("WATCHER", "SpiritShield+1"),
    ("WATCHER", "Deep Breath"),
    ("WATCHER", "InnerPeace"),
    ("WATCHER", "Dark Shackles"),
    ("WATCHER", "Impatience"),
    ("WATCHER", "Panacea"),
    ("WATCHER", "Wallop"),
    ("WATCHER", "Adaptation+1"),
    ("WATCHER", "WheelKick+1"),
    ("WATCHER", "Ragnarok+1"),
    ("WATCHER", "Ragnarok"),
    ("WATCHER", "Tantrum"),
    ("WATCHER", "Sanctity+1"),
    ("WATCHER", "Sanctity"),
    ("WATCHER", "EmptyMind"),
], columns=["character_chosen", "card_name"])

print("Cards of interest:", len(CARDS_OF_INTEREST))

In [ ]:
candidate_characters = CARDS_OF_INTEREST["character_chosen"].unique().tolist()
candidate_cards = CARDS_OF_INTEREST["card_name"].unique().tolist()

# Coarse filter in Spark (character/card lists, not exact pairs — colorless cards can appear
# for multiple characters), collect once, then narrow to exact (character, card) pairs and fit
# regressions in pandas. Avoids re-querying the full table once per candidate card.
regression_pd = (
    df.filter(F.col("character_chosen").isin(candidate_characters) & F.col("card_name").isin(candidate_cards))
    .select("character_chosen", "card_name", "was_picked", "victory", "floor", "current_hp", "max_hp", "relic_count", "ascension_level")
    .na.drop()
    .toPandas()
)
regression_pd = regression_pd.merge(CARDS_OF_INTEREST, on=["character_chosen", "card_name"], how="inner")
print("Collected rows for regression candidates:", len(regression_pd))

In [ ]:
regression_pd["was_picked"] = regression_pd["was_picked"].astype(int)
regression_pd["victory"] = regression_pd["victory"].astype(int)
regression_pd = regression_pd[regression_pd["max_hp"] > 0].copy()
regression_pd["hp_ratio"] = regression_pd["current_hp"] / regression_pd["max_hp"]

MIN_REGRESSION_ROWS = 200

# groupby().apply() needs every group to return a Series with the same keys — a group
# returning fewer keys than another (e.g. just n/error on failure) makes pandas fall back
# to a stacked long-format result instead of one row per group, so every branch below
# fills the full set of keys even when most are None.
EMPTY_RESULT = {
    "n": None,
    "error": None,
    "was_picked_coef": None,
    "was_picked_pvalue": None,
    "odds_ratio": None,
    "odds_ratio_ci_low": None,
    "odds_ratio_ci_high": None,
}

def fit_card_logit(group):
    result = dict(EMPTY_RESULT)
    if len(group) < MIN_REGRESSION_ROWS:
        result["n"] = len(group)
        result["error"] = "too few rows"
        return pd.Series(result)
    try:
        model = smf.logit(
            "victory ~ was_picked + hp_ratio + floor + relic_count + ascension_level",
            data=group,
        ).fit(disp=0)
    except Exception as exc:
        result["n"] = len(group)
        result["error"] = str(exc)
        return pd.Series(result)
    coef = model.params["was_picked"]
    ci_low, ci_high = model.conf_int().loc["was_picked"]
    result.update({
        "n": len(group),
        "error": None,
        "was_picked_coef": coef,
        "was_picked_pvalue": model.pvalues["was_picked"],
        "odds_ratio": np.exp(coef),
        "odds_ratio_ci_low": np.exp(ci_low),
        "odds_ratio_ci_high": np.exp(ci_high),
    })
    return pd.Series(result)

results_pd = (
    regression_pd.groupby(["character_chosen", "card_name"])
    .apply(fit_card_logit, include_groups=False)
    .reset_index()
)
print("Fitted:", (results_pd["error"].isna()).sum(), "of", len(results_pd))

### Results

`odds_ratio` > 1 means picking the card is associated with higher win odds after controlling for HP ratio, floor, relic count, and ascension at the time it was offered; < 1 means lower. Sorted within each character by odds ratio, restricted to statistically significant results (`was_picked_pvalue < 0.05`) — cards that didn't reach significance at this sample size are dropped from this view but still available in `results_pd`.

In [ ]:
significant = results_pd[
    results_pd["error"].isna() & (results_pd["was_picked_pvalue"] < 0.05)
].sort_values(["character_chosen", "odds_ratio"], ascending=[True, False])

cols = ["character_chosen", "card_name", "n", "odds_ratio", "odds_ratio_ci_low", "odds_ratio_ci_high", "was_picked_pvalue"]
significant[cols]

## Stop Spark

Run this when done exploring — otherwise the JVM stays alive holding memory until the kernel is restarted.

In [ ]:
spark.stop()